# 可选实验：逻辑回归的梯度下降

## 目标
在本实验中，你将：
- 更新逻辑回归的梯度下降。
- 在熟悉的数据集上探索梯度下降

In [1]:
import copy, math
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from lab_utils_common import  dlc, plot_data, plt_tumor_data, sigmoid, compute_cost_logistic
from plt_quad_logistic import plt_quad_logistic, plt_prob
plt.style.use('./deeplearning.mplstyle')

## 数据集
让我们从决策边界实验中使用的相同双特征数据集开始。

In [2]:
X_train = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y_train = np.array([0, 0, 0, 1, 1, 1])

和之前一样，我们将使用辅助函数来绘制这些数据。标签为 $y=1$ 的数据点显示为红色叉号，而标签为 $y=0$ 的数据点显示为蓝色圆圈。

In [3]:
fig,ax = plt.subplots(1,1,figsize=(4,4))
plot_data(X_train, y_train, ax)

ax.axis([0, 4, 0, 3.5])
ax.set_ylabel('$x_1$', fontsize=12)
ax.set_xlabel('$x_0$', fontsize=12)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## 逻辑梯度下降
<img align="right" src="./images/C1_W3_Logistic_gradient_descent.png"     style=" width:400px; padding: 10px; " >

回顾梯度下降算法使用梯度计算：
$$\begin{align*}
&\text{重复直到收敛:} \; \lbrace \\
&  \; \; \;w_j = w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{1}  \; & \text{对于 j := 0..n-1} \\ 
&  \; \; \;  \; \;b = b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b} \\
&\rbrace
\end{align*}$$

其中每次迭代对所有 $j$ 的 $w_j$ 进行同步更新，其中
$$\begin{align*}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \tag{2} \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{3} 
\end{align*}$$

* m 是数据集中的训练样本数量
* $f_{\mathbf{w},b}(x^{(i)})$ 是模型的预测值，而 $y^{(i)}$ 是目标值
* 对于逻辑回归模型
    $z = \mathbf{w} \cdot \mathbf{x} + b$  
    $f_{\mathbf{w},b}(x) = g(z)$  
    其中 $g(z)$ 是 sigmoid 函数：
    $g(z) = \frac{1}{1+e^{-z}}$   
    


### 梯度下降实现
梯度下降算法实现有两个组件：
- 实现上面方程(1)的循环。这是下面的 `gradient_descent`，通常在可选和练习实验中提供给你。
- 计算当前梯度，即上面的方程(2,3)。这是下面的 `compute_gradient_logistic`。你将在本周的练习实验中被要求实现它。

#### 计算梯度，代码描述
实现上面所有 $w_j$ 和 $b$ 的方程(2),(3)。
有很多方法可以实现这一点。下面概述的是：
- 初始化变量以累加 `dj_dw` 和 `dj_db`
- 对于每个样本
    - 计算该样本的误差 $g(\mathbf{w} \cdot \mathbf{x}^{(i)} + b) - \mathbf{y}^{(i)}$
    - 对于该样本中的每个输入值 $x_{j}^{(i)}$，
        - 将误差乘以输入 $x_{j}^{(i)}$，并加到 `dj_dw` 的相应元素上。（上面的方程2）
    - 将误差加到 `dj_db` 上（上面的方程3）

- 将 `dj_db` 和 `dj_dw` 除以总样本数 (m)
- 注意 numpy 中的 $\mathbf{x}^{(i)}$ 是 `X[i,:]` 或 `X[i]`，而 $x_{j}^{(i)}$ 是 `X[i,j]`

In [4]:
def compute_gradient_logistic(X, y, w, b): 
    """
    计算线性回归的梯度
 
    参数:
      X (ndarray (m,n): 数据，m个样本，每个样本有n个特征
      y (ndarray (m,)): 目标值
      w (ndarray (n,)): 模型参数
      b (标量)      : 模型参数
    返回:
      dj_dw (ndarray (n,)): 代价相对于参数w的梯度
      dj_db (标量)      : 代价相对于参数b的梯度
    """
    m,n = X.shape
    dj_dw = np.zeros((n,))                           #(n,)
    dj_db = 0.

    for i in range(m):
        f_wb_i = sigmoid(np.dot(X[i],w) + b)          #(n,)(n,)=标量
        err_i  = f_wb_i  - y[i]                       #标量
        for j in range(n):
            dj_dw[j] = dj_dw[j] + err_i * X[i,j]      #标量
        dj_db = dj_db + err_i
    dj_dw = dj_dw/m                                   #(n,)
    dj_db = dj_db/m                                   #标量
        
    return dj_db, dj_dw  

使用下面的单元格检查梯度函数的实现。

In [5]:
X_tmp = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y_tmp = np.array([0, 0, 0, 1, 1, 1])
w_tmp = np.array([2.,3.])
b_tmp = 1.
dj_db_tmp, dj_dw_tmp = compute_gradient_logistic(X_tmp, y_tmp, w_tmp, b_tmp)
print(f"dj_db: {dj_db_tmp}" )
print(f"dj_dw: {dj_dw_tmp.tolist()}" )

dj_db: 0.49861806546328574
dj_dw: [0.498333393278696, 0.49883942983996693]


**期望输出**
``` 
dj_db: 0.49861806546328574
dj_dw: [0.498333393278696, 0.49883942983996693]
```

#### 梯度下降代码
实现上面方程(1)的代码在下面实现。花点时间找到并将例程中的函数与上面的方程进行比较。

In [6]:
def gradient_descent(X, y, w_in, b_in, alpha, num_iters): 
    """
    执行批量梯度下降
    
    参数:
      X (ndarray (m,n)   : 数据，m个样本，每个样本有n个特征
      y (ndarray (m,))   : 目标值
      w_in (ndarray (n,)): 模型参数的初始值
      b_in (标量)      : 模型参数的初始值
      alpha (浮点数)      : 学习率
      num_iters (标量) : 运行梯度下降的迭代次数
      
    返回:
      w (ndarray (n,))   : 更新后的参数值
      b (标量)         : 更新后的参数值
    """
    # 一个数组，用于存储每次迭代的代价J和w，主要用于后续绘图
    J_history = []
    w = copy.deepcopy(w_in)  # 避免在函数内修改全局w
    b = b_in
    
    for i in range(num_iters):
        # 计算梯度并更新参数
        dj_db, dj_dw = compute_gradient_logistic(X, y, w, b)   

        # 使用w、b、alpha和梯度更新参数
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db               
      
        # 每次迭代保存代价J
        if i<100000:      # 防止资源耗尽
            J_history.append( compute_cost_logistic(X, y, w, b) )

        # 每隔一定间隔打印代价，如果迭代次数<10则打印10次
        if i% math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]}   ")
        
    return w, b, J_history         # 返回最终的w、b和J历史用于绘图


让我们在数据集上运行梯度下降。

In [7]:
w_tmp  = np.zeros_like(X_train[0])
b_tmp  = 0.
alph = 0.1
iters = 10000

w_out, b_out, _ = gradient_descent(X_train, y_train, w_tmp, b_tmp, alph, iters) 
print(f"\nupdated parameters: w:{w_out}, b:{b_out}")

Iteration    0: Cost 0.684610468560574   
Iteration 1000: Cost 0.1590977666870456   
Iteration 2000: Cost 0.08460064176930081   
Iteration 3000: Cost 0.05705327279402531   
Iteration 4000: Cost 0.042907594216820076   
Iteration 5000: Cost 0.034338477298845684   
Iteration 6000: Cost 0.028603798022120097   
Iteration 7000: Cost 0.024501569608793   
Iteration 8000: Cost 0.02142370332569295   
Iteration 9000: Cost 0.019030137124109114   

updated parameters: w:[5.28 5.08], b:-14.222409982019837


#### 让我们绘制梯度下降的结果：

In [8]:
fig,ax = plt.subplots(1,1,figsize=(5,4))
# 绘制概率
plt_prob(ax, w_out, b_out)

# 绘制原始数据
ax.set_ylabel(r'$x_1$')
ax.set_xlabel(r'$x_0$')   
ax.axis([0, 4, 0, 3.5])
plot_data(X_train,y_train,ax)

# 绘制决策边界
x0 = -b_out/w_out[0]
x1 = -b_out/w_out[1]
ax.plot([0,x0],[x1,0], c=dlc["dlblue"], lw=1)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

在上图中：
 - 阴影反映了 y=1 的概率（决策边界之前的结果）
 - 决策边界是概率 = 0.5 的线
 

## 另一个数据集
让我们回到单变量数据集。只有两个参数，$w$、$b$，可以使用等高线图绘制代价函数，以更好地了解梯度下降的作用。

In [9]:
x_train = np.array([0., 1, 2, 3, 4, 5])
y_train = np.array([0,  0, 0, 1, 1, 1])

和之前一样，我们将使用辅助函数来绘制这些数据。标签为 $y=1$ 的数据点显示为红色叉号，而标签为 $y=0$ 的数据点显示为蓝色圆圈。

In [10]:
fig,ax = plt.subplots(1,1,figsize=(4,3))
plt_tumor_data(x_train, y_train, ax)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

在下面的图中，尝试：
- 通过点击右上角的等高线图来更改 $w$ 和 $b$。
    - 更改可能需要一两秒钟
    - 注意左上图中代价的变化值。
    - 注意代价是通过每个样本的损失累加的（垂直虚线）
- 通过点击橙色按钮运行梯度下降。
    - 注意代价稳步下降（等高线和代价图采用 log(cost)）
    - 在等高线图中点击将重置模型以进行新的运行
- 要重置图表，请重新运行单元格

In [11]:
w_range = np.array([-1, 7])
b_range = np.array([1, -14])
quad = plt_quad_logistic( x_train, y_train, w_range, b_range )

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## 恭喜！
你已经：
- 检验了计算逻辑回归梯度的公式和实现
- 利用这些例程
    - 探索了单变量数据集
    - 探索了双变量数据集